# Notebook 04: Wind turbine footprint

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

# I define the folders for my wind analysis.
ROOT = Path(r"G:/My Drive/Projects/Large projects/2026TexasEnergyFootprint")
INPUT = ROOT / "05_Optimized_Data" / "GeoParquet"
OUTPUT = ROOT / "06_Energy_Footprint_Analysis" / "Wind"
OUTPUT.mkdir(parents=True, exist_ok=True)

# I define the unit conversions used in my calculations.
ACRE_M2 = 4046.8564224   # Square meters in one acre
FT_PER_MILE = 5280.0      # Feet in one mile
SQFT_PER_ACRE = 43560.0   # Square feet in one acre

# I use the mean permanent direct land-impact coefficient
# reported by Harrison-Atlas et al. (2022).
WIND_HA_PER_MW = 0.65

# I load the cleaned geographic datasets.
def read_layer(name):
    return gpd.read_parquet(INPUT / f"{name}.parquet")

# I standardize Texas county GEOIDs to five digits.
def county_key(df):
    return (
        df["county_geoid"]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

# I save one final footprint estimate at each geographic level.
def save_results(detail, county, state, assumptions):

    # I verify that the facility and statewide acreage agree.
    assert np.isclose(
        detail["wind_acres"].sum(),
        state["wind_acres"].sum(),
        atol=0.01
    )

    # I save the final results.
    detail.drop(columns="geometry", errors="ignore").to_csv(
        OUTPUT / "facility_footprint.csv", index=False
    )
    county.to_csv(OUTPUT / "county_footprint.csv", index=False)
    state.to_csv(OUTPUT / "statewide_footprint.csv", index=False)
    assumptions.to_csv(OUTPUT / "footprint_assumptions.csv", index=False)

    print("Results saved to:", OUTPUT)
    display(state.round(2))
    display(county.head(10).round(2))

In [2]:
# I load the cleaned wind turbine data from Notebook 03.
wind = read_layer("Texas_Wind_Turbines")

# I check the turbine count and unique identifiers.
assert len(wind) == 19943
assert wind["turbine_id"].is_unique

# I standardize county IDs and convert capacity from kW to MW.
wind["county_geoid"] = county_key(wind)
wind["capacity_mw"] = pd.to_numeric(
    wind["capacity_kw"], errors="coerce"
) / 1000

# I identify turbines with missing or invalid capacity.
# I keep these turbines and estimate their capacity in the next step.
missing_capacity = wind["capacity_mw"].isna() | (wind["capacity_mw"] <= 0)

print("Total turbines:", f"{len(wind):,}")
print("Valid capacity:", f"{(~missing_capacity).sum():,}")
print("Missing capacity:", f"{missing_capacity.sum():,}")
print("Missing percentage:", f"{missing_capacity.mean() * 100:.2f}%")

Total turbines: 19,943
Valid capacity: 19,204
Missing capacity: 739
Missing percentage: 3.71%


In [ ]:
# I identified 739 turbines (3.71%) with missing or invalid capacity.
# I keep these turbines to avoid underestimating the wind footprint.
#
# I estimate their capacity in two steps:
# 1. I first use the capacity of other turbines in the same project,
#    but only when at least three turbines have known capacities
#    and their capacities are nearly identical (within 0.001 MW).
# 2. If project-level estimation is not possible, I use the
#    statewide median capacity of turbines with valid data.

In [3]:
# I identify turbines with missing or nonpositive capacity.
wind["capacity_missing"] = (
    wind["capacity_mw"].isna()
    | wind["capacity_mw"].le(0)
)

missing_wind = wind.loc[wind["capacity_missing"]].copy()
valid_wind = wind.loc[~wind["capacity_missing"]].copy()

# I examine which fields are available for estimating capacity.
print("Available wind columns:")
print(wind.columns.tolist())

print("\nCAPACITY COMPLETENESS")
print("Total turbines:", f"{len(wind):,}")
print("Valid capacity:", f"{len(valid_wind):,}")
print("Missing/nonpositive capacity:", f"{len(missing_wind):,}")
print(
    "Percent missing:",
    f"{len(missing_wind) / len(wind) * 100:.2f}%"
)

# I check whether missing capacities are concentrated in
# particular projects, manufacturers, years, or counties.
review_columns = [
    "turbine_id",
    "p_name",
    "p_year",
    "t_manu",
    "t_model",
    "capacity_kw",
    "capacity_mw",
    "county_geoid"
]

available_columns = [
    col for col in review_columns
    if col in wind.columns
]

print("\nSAMPLE RECORDS WITH MISSING CAPACITY")
display(missing_wind[available_columns].head(20))

for col in ["p_name", "p_year", "t_manu", "t_model"]:
    if col in missing_wind.columns:
        print(f"\nMISSING CAPACITY BY {col}")
        display(
            missing_wind[col]
            .value_counts(dropna=False)
            .head(15)
            .rename_axis(col)
            .reset_index(name="turbines")
        )

# I compare known turbine capacities before considering imputation.
print("\nVALID TURBINE CAPACITY DISTRIBUTION (MW)")
display(
    valid_wind["capacity_mw"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

Available wind columns:
['turbine_id', 'project_name', 'installation_year', 't_manu', 't_model', 'capacity_kw', 't_hh', 't_rd', 'county_geoid', 'county_name', 'county_match_status', 'source_row_id', 'geometry', 'capacity_mw', 'capacity_missing']

CAPACITY COMPLETENESS
Total turbines: 19,943
Valid capacity: 19,204
Missing/nonpositive capacity: 739
Percent missing: 3.71%

SAMPLE RECORDS WITH MISSING CAPACITY


,turbine_id,t_manu,t_model,capacity_kw,capacity_mw,county_geoid
4235,3134819,None,None,NaN,NaN,48133
6310,3139468,None,None,NaN,NaN,48437
6311,3142078,None,None,NaN,NaN,48069
6312,3142141,None,None,NaN,NaN,48069
6313,3139904,None,None,NaN,NaN,48437
6314,3139447,None,None,NaN,NaN,48437
6315,3140325,None,None,NaN,NaN,48069
6316,3140204,None,None,NaN,NaN,48069
6317,3139793,None,None,NaN,NaN,48437
6318,3139988,None,None,NaN,NaN,48069



MISSING CAPACITY BY t_manu


,t_manu,turbines
0,None,739



MISSING CAPACITY BY t_model


,t_model,turbines
0,None,739



VALID TURBINE CAPACITY DISTRIBUTION (MW)


count    19204.000000
mean         2.287459
std          0.881220
min          0.060000
10%          1.500000
25%          1.700000
50%          2.200000
75%          2.820000
90%          3.450000
max          6.000000
Name: capacity_mw, dtype: float64

In [4]:

# I preserve the original turbine capacities before estimating missing values.
wind["capacity_mw_original"] = wind["capacity_mw"]

# I identify turbines with observed and missing capacity.
wind["capacity_source"] = np.where(
    wind["capacity_mw"] > 0, "Observed", "Missing"
)

# I standardize project names to match turbines within the same project.
wind["project_key"] = (
    wind["project_name"].astype("string").str.strip().str.lower()
    .replace(["", "none", "nan", "<na>"], pd.NA)
)

# I calculate capacity statistics for turbines with known capacity.
project_capacity = (
    wind.loc[wind["capacity_mw"].gt(0) & wind["project_key"].notna()]
    .groupby("project_key")["capacity_mw"]
    .agg(["size", "median", "min", "max"])
)

# I use project capacity only when at least three turbines
# have nearly identical capacities (within 0.001 MW).
consistent = project_capacity[
    (project_capacity["size"] >= 3)
    & np.isclose(
        project_capacity["min"],
        project_capacity["max"],
        atol=0.001
    )
]

# I estimate missing capacity using consistent project values.
wind["capacity_mw_final"] = wind["capacity_mw"].where(
    wind["capacity_mw"] > 0
)

project_estimate = wind["project_key"].map(consistent["median"])

project_match = (
    wind["capacity_mw_final"].isna()
    & project_estimate.notna()
)

wind.loc[project_match, "capacity_mw_final"] = project_estimate[project_match]
wind.loc[project_match, "capacity_source"] = "Estimated from project"

# I use the statewide median for turbines without a project estimate.
statewide_median = wind.loc[
    wind["capacity_mw"].gt(0), "capacity_mw"
].median()

remaining = wind["capacity_mw_final"].isna()

wind.loc[remaining, "capacity_mw_final"] = statewide_median
wind.loc[remaining, "capacity_source"] = "Estimated from statewide median"

# I check that every turbine now has a valid capacity.
assert len(wind) == 19943
assert wind["capacity_mw_final"].gt(0).all()

# I summarize how missing capacities were estimated.
capacity_audit = (
    wind.groupby("capacity_source")
    .agg(
        turbines=("turbine_id", "size"),
        capacity_mw=("capacity_mw_final", "sum")
    )
    .reset_index()
)

# I save the final capacities and the estimation audit.
wind.drop(columns="geometry").to_csv(
    OUTPUT / "wind_turbine_capacity_final.csv", index=False
)

capacity_audit.to_csv(
    OUTPUT / "wind_capacity_imputation_audit.csv", index=False
)

print("CAPACITY ESTIMATION COMPLETE")
print("Total turbines:", f"{len(wind):,}")
print("Statewide median capacity (MW):", round(statewide_median, 2))
print("Total estimated capacity (MW):", round(wind["capacity_mw_final"].sum(), 3))

display(capacity_audit)

CAPACITY ESTIMATION COMPLETE
Total turbines: 19,943
Statewide median capacity (MW): 2.2
Total estimated capacity (MW): 45554.165


,capacity_source,turbines,capacity_mw
0,Estimated from statewide median,739,1625.800
1,Observed,19204,43928.365


## Assumptions and statewide calculation

I estimate **permanent direct land impact** using **0.65 hectares per MW**, reported by Harrison-Atlas, Lopez, and Lantz (2022), *Dynamic land use implications of rapidly expanding and evolving wind power deployment*, *Environmental Research Letters* 17, 044064, Section 2.5, printed page 8 (PDF page 9). DOI: https://doi.org/10.1088/1748-9326/ac5f2c.

The paper reports this **mean** from Diffendorfer et al. (2019), *Geographic context affects the landscape change and fragmentation caused by wind energy facilities*, *PeerJ* 7, e7129. It covers permanent direct impact, not temporary construction disturbance or the entire area spanned by turbines.

I use **one land-impact coefficient only**.

In [5]:
# I use the mean permanent direct land-impact coefficient of
# 0.65 hectares per MW reported in Harrison-Atlas et al. (2022),
# Section 2.5, based on Diffendorfer et al. (2019).
#
# I estimate permanent direct land impact only.
# I exclude temporary construction disturbance and turbine spacing.

SOURCE_URL = "https://doi.org/10.1088/1748-9326/ac5f2c"
PRIMARY_SOURCE_URL = "https://doi.org/10.7717/peerj.7129"

HECTARES_PER_MW = 0.65
PERMANENT_ACRES_PER_MW = HECTARES_PER_MW * 10000 / ACRE_M2

print(
    "Permanent land-impact coefficient:",
    f"{PERMANENT_ACRES_PER_MW:.4f} acres/MW"
)

Permanent land-impact coefficient: 1.6062 acres/MW


In [6]:
# I document the coefficient used to estimate permanent wind land impact.
# I use a U.S. study-derived estimate, not a Texas-specific measurement.
# I exclude temporary construction disturbance and turbine spacing.

wind_assumptions = pd.DataFrame([{
    "impact_type": "Permanent direct land impact",
    "hectares_per_mw": HECTARES_PER_MW,
    "acres_per_mw": PERMANENT_ACRES_PER_MW,
    "source": "Harrison-Atlas et al. (2022), Section 2.5",
    "source_url": SOURCE_URL,
    "underlying_source_url": PRIMARY_SOURCE_URL,
    "notes": "Mean permanent direct impact; temporary disturbance excluded"
}])

In [8]:
# I save the assumptions used to calculate the wind footprint.
wind_assumptions.to_csv(
    OUTPUT / "wind_acreage_assumptions.csv",
    index=False
)

In [9]:
# I calculate permanent direct land impact using the final turbine capacity.
# I exclude temporary construction disturbance and turbine spacing.
wind["wind_acres"] = (
    wind["capacity_mw_final"] * PERMANENT_ACRES_PER_MW
)

# I identify turbines whose capacities were estimated.
wind["capacity_estimated"] = wind["capacity_source"].ne("Observed")

# I standardize county identifiers for aggregation.
wind["county_geoid"] = county_key(wind)

# I load the official Census county reference for Texas.
census_url = (
    "https://www2.census.gov/geo/docs/reference/codes2020/"
    "national_county2020.txt"
)

counties = pd.read_csv(census_url, sep="|", dtype=str)

counties = counties.loc[
    counties["STATEFP"] == "48",
    ["STATEFP", "COUNTYFP", "COUNTYNAME"]
].copy()

counties["county_geoid"] = counties["STATEFP"] + counties["COUNTYFP"]
counties = counties.rename(
    columns={"COUNTYNAME": "county_name"}
)[["county_geoid", "county_name"]]

# I check that the reference contains all 254 Texas counties.
assert len(counties) == 254
assert counties["county_geoid"].is_unique

# I verify that assigned county identifiers match the reference.
invalid_counties = (
    set(wind["county_geoid"].dropna())
    - set(counties["county_geoid"])
)
assert not invalid_counties

# I summarize turbine counts, estimated capacities and acreage by county.
county_summary = (
    wind.groupby("county_geoid", as_index=False)
    .agg(
        turbines=("turbine_id", "size"),
        estimated_capacity_turbines=("capacity_estimated", "sum"),
        capacity_mw=("capacity_mw_final", "sum"),
        wind_acres=("wind_acres", "sum")
    )
)

# I retain all 254 counties, including those without turbines.
wind_county_254 = counties.merge(
    county_summary,
    on="county_geoid",
    how="left",
    validate="one_to_one"
)

numeric_cols = [
    "turbines", "estimated_capacity_turbines",
    "capacity_mw", "wind_acres"
]

wind_county_254[numeric_cols] = wind_county_254[numeric_cols].fillna(0)

wind_county_254[
    ["turbines", "estimated_capacity_turbines"]
] = wind_county_254[
    ["turbines", "estimated_capacity_turbines"]
].astype(int)

wind_county_254 = (
    wind_county_254.sort_values("county_name").reset_index(drop=True)
)

# I retain turbines without county assignments for reconciliation.
wind_unassigned = wind[wind["county_geoid"].isna()].copy()

# I calculate the statewide wind footprint.
wind_state = pd.DataFrame([{
    "sector": "Wind",
    "turbines": len(wind),
    "estimated_capacity_turbines": int(wind["capacity_estimated"].sum()),
    "capacity_mw": wind["capacity_mw_final"].sum(),
    "wind_acres": wind["wind_acres"].sum()
}])

# I verify that county totals and unassigned turbines
# reconcile with the statewide inventory and acreage.
assert (
    wind_county_254["turbines"].sum() + len(wind_unassigned)
    == len(wind)
)

for col, source_col in [
    ("capacity_mw", "capacity_mw_final"),
    ("wind_acres", "wind_acres")
]:
    assert np.isclose(
        wind_county_254[col].sum() + wind_unassigned[source_col].sum(),
        wind[source_col].sum(),
        atol=0.01
    )

# I save the county and statewide results.
wind_county_254.to_csv(
    OUTPUT / "wind_county_254_with_names.csv", index=False
)

wind_state.to_csv(
    OUTPUT / "wind_statewide_acreage.csv", index=False
)

print("Texas counties:", len(wind_county_254))
print("Total turbines:", f"{len(wind):,}")
print("Unassigned turbines:", len(wind_unassigned))
print("Total capacity (MW):", f"{wind_state['capacity_mw'].iloc[0]:,.3f}")
print("Permanent wind acreage:", f"{wind_state['wind_acres'].iloc[0]:,.2f}")

display(wind_state)
display(wind_county_254.head(10))

Texas counties: 254
Total turbines: 19,943
Unassigned turbines: 0
Total capacity (MW): 45,554.165
Permanent wind acreage: 73,168.42


,sector,turbines,estimated_capacity_turbines,capacity_mw,wind_acres
0,Wind,19943,739,45554.165,73168.415578


,county_geoid,county_name,turbines,estimated_capacity_turbines,capacity_mw,wind_acres
0,48001,Anderson County,0,0,0.00,0.000000
1,48003,Andrews County,57,0,160.74,258.178174
2,48005,Angelina County,0,0,0.00,0.000000
3,48007,Aransas County,0,0,0.00,0.000000
4,48009,Archer County,188,0,358.40,575.656697
5,48011,Armstrong County,93,2,328.90,528.274240
6,48013,Atascosa County,0,0,0.00,0.000000
7,48015,Austin County,0,0,0.00,0.000000
8,48017,Bailey County,35,0,120.75,193.946836
9,48019,Bandera County,0,0,0.00,0.000000


In [9]:
## Data checker the next cell is for hexagon notebook 

In [10]:
# I use the original mosaic grid to preserve consistent hexagon boundaries.
MOSAIC = ROOT / "07_Visualization" / "03_Texas_Energy_Land_Mosaic"

hex_grid = gpd.read_parquet(
    MOSAIC / "Texas_10sqmi_Hex_Grid.parquet"
)

# I use the final turbine capacities and permanent land-impact estimates.
wind_points = wind[
    ["turbine_id", "capacity_mw_final", "wind_acres", "geometry"]
].to_crs(hex_grid.crs)

# I assign each turbine to the hexagon containing its location.
joined = gpd.sjoin(
    wind_points,
    hex_grid[["hex_id", "geometry"]],
    how="left",
    predicate="within"
)

# I check that every turbine belongs to exactly one hexagon.
assert joined["hex_id"].notna().all()
assert joined["turbine_id"].is_unique
assert len(joined) == len(wind)

# I calculate turbine counts, capacity and permanent acreage per hexagon.
wind_hex_summary = (
    joined.groupby("hex_id", as_index=False)
    .agg(
        wind_turbines=("turbine_id", "size"),
        wind_capacity_mw=("capacity_mw_final", "sum"),
        wind_acres=("wind_acres", "sum")
    )
)

# I retain all hexagons, including those without wind turbines.
wind_hex_revised = hex_grid[
    ["hex_id", "geometry", "area_sq_miles", "is_partial"]
].merge(
    wind_hex_summary,
    on="hex_id",
    how="left",
    validate="one_to_one"
)

cols = ["wind_turbines", "wind_capacity_mw", "wind_acres"]
wind_hex_revised[cols] = wind_hex_revised[cols].fillna(0)
wind_hex_revised["wind_turbines"] = wind_hex_revised["wind_turbines"].astype(int)

# I calculate the modeled wind footprint as a percentage of hexagon area.
wind_hex_revised["wind_fraction"] = (
    wind_hex_revised["wind_acres"]
    / (wind_hex_revised.geometry.area / ACRE_M2)
)
wind_hex_revised["wind_coverage_pct"] = wind_hex_revised["wind_fraction"] * 100

# I reconcile the hexagon results with the original turbine data.
assert len(wind_hex_revised) == len(hex_grid)
assert wind_hex_revised["wind_turbines"].sum() == len(wind)

assert np.isclose(
    wind_hex_revised["wind_acres"].sum(),
    wind["wind_acres"].sum(),
    atol=0.01
)

# I save the final wind hexagon layer for the mosaic.
wind_hex_revised.to_parquet(
    MOSAIC / "Texas_Wind_Hex_Final_Revised.parquet"
)

print("Hexagons:", f"{len(wind_hex_revised):,}")
print("Turbines:", f"{wind_hex_revised['wind_turbines'].sum():,}")
print("Permanent wind acres:", f"{wind_hex_revised['wind_acres'].sum():,.2f}")

Hexagons: 27,421
Turbines: 19,943
Permanent wind acres: 73,168.42
